# 05 · Detección híbrida OpenCV + Slim-CNN

Este notebook recibe una foto con varios granos sobre fondo blanco/claro. OpenCV propone una región para cada componente separado; Slim-CNN clasifica cada recorte; al final se dibuja una caja y su clase sobre la foto original.

**Requisitos:** ejecuta primero el entrenamiento Slim-CNN en `04_entrenamiento_modelos.ipynb` para crear `checkpoints/slim_cnn/best.pt`, y proporciona una foto de varios granos. Este pipeline no es un detector YOLO aprendido: la localización la hace OpenCV. Granos pegados pueden salir como una sola región.


In [ ]:
from pathlib import Path
import sys
import cv2
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise FileNotFoundError('Abre el notebook desde la carpeta del proyecto.')
sys.path.insert(0, str(ROOT))
from src.inference.multi_bean import (MultiBeanSlimCNN, segment_beans,
    analyze_multi_bean_image, draw_candidate_boxes, draw_predictions)
IMAGE_PATH = ROOT / 'data' / 'inference_examples' / 'varios_granos.jpg'
CHECKPOINT = ROOT / 'checkpoints' / 'slim_cnn' / 'best.pt'
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f'Copia una foto de varios granos a {IMAGE_PATH.parent} y actualiza IMAGE_PATH.')
image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise ValueError(f'OpenCV no pudo leer {IMAGE_PATH}')
print('Imagen cargada:', IMAGE_PATH)
print('Checkpoint Slim-CNN disponible:', CHECKPOINT.is_file())


## Ajustar y revisar segmentación

Los parámetros de área ayudan a quitar puntos pequeños y componentes que parecen grupos de granos. La máscara en blanco representa lo que OpenCV tomó como primer plano. Revisa que haya aproximadamente una caja por grano antes de interpretar las clases.


In [ ]:
MIN_AREA = 0.0001
MAX_AREA = 0.30
components, mask = segment_beans(image, MIN_AREA, MAX_AREA)
candidates = draw_candidate_boxes(image, components)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); axes[0].set_title('Foto original')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title(f'Máscara OpenCV ({len(components)} regiones)')
axes[2].imshow(cv2.cvtColor(candidates, cv2.COLOR_BGR2RGB)); axes[2].set_title('Cajas candidatas de OpenCV')
for ax in axes: ax.axis('off')
plt.tight_layout()


## Inspeccionar recortes

Si una caja cubre más de un grano o corta un grano, ajusta la segmentación o mejora la toma de la foto antes de confiar en la etiqueta. El modelo solo puede clasificar correctamente los recortes que OpenCV separó bien.


In [ ]:
if not CHECKPOINT.is_file():
    print('Para ver etiquetas falta entrenar Slim-CNN en 04_entrenamiento_modelos.ipynb.')
else:
    classifier = MultiBeanSlimCNN(CHECKPOINT)
    predictions, _ = analyze_multi_bean_image(image, classifier, MIN_AREA, MAX_AREA)
    annotated = draw_predictions(image, predictions)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)); axes[0].set_title(f'Resultado híbrido ({len(predictions)} granos)')
    if predictions:
        axes[1].axis('off')
        print('Clases y puntajes softmax (no calibrados):')
        for i, result in enumerate(predictions, 1):
            print(f'{i}: {result.label}, score={result.confidence:.3f}, caja={result.bbox}')
    else:
        axes[1].text(0.05, 0.5, 'No se encontraron granos', transform=axes[1].transAxes)
    plt.tight_layout()


Cuando el resultado visual sea razonable, el mismo flujo puede ejecutarse fuera del notebook: `python -m src.predict_multi_beans --source ruta/foto.jpg`. Para dibujar solo defectos: `--only defecto`. Para una medición científica, hace falta crear un conjunto de fotos de escenas múltiples con cajas y clases revisadas a mano; reportar por separado cuántos granos se segmentaron correctamente y la exactitud de clasificación de los recortes correctos.
